# Criteo Train

Minimal path to train an XGBoost CTR model: load + encode data, then either a single fit or a wandb hyperparameter sweep.

For exploratory analysis, frequent-itemset/association-rule mining, and tree inspection, see `criteo_analysis.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xgboost as xgb
from sklearn.metrics import log_loss
from wandb.integration.xgboost import WandbCallback

import wandb
from criteo_functions import *

## Load data

In [ ]:
data_dir = get_data_dir()

config = {
    "sample_every": 1,
    "cat_cutoff": 100,
    "cat_pseudo_count": 16,
    "cat_order": None,  # just needed for target encoding or not
}

(train_X, train_y), (valid_X, valid_y), (test_X, test_y) = get_data(config, data_dir)
train_X.shape, valid_X.shape, test_X.shape

In [ ]:
def log_lossv(target, pred):
    # lt = np.empty_like(target, dtype=np.float64)
    # ll = -np.log(pred, out=llwhere=target), np.log(1-pred))
    ll = -np.log(np.where(target, pred, (1 - pred)))
    return ll

In [ ]:
data_ys = {
    "train": train_y,
    "valid": valid_y,
    "test": test_y,
}
stats = {}
for key, data_y in data_ys.items():
    stat = {}
    stat["y_mean"] = data_y.mean()
    stat["y_count"] = len(data_y)
    stat["y_std"] = data_y.std()
    stat["y_std binom"] = np.sqrt(stat["y_mean"] * (1 - stat["y_mean"]))

    stat["ll_mean"] = log_lossv(data_y, stat["y_mean"]).mean()
    stat["ll_mean_std"] = log_lossv(data_y, stat["y_mean"]).std()
    stat["ll_mean_se"] = stat["ll_mean_std"] / np.sqrt(stat["y_count"])
    print(key, stat)
    stats[key] = stat

## Train (single fit)

Hyperparameters below are the best found in the `criteo_boost_sweep` sweep (see `wandb.md`): `learning_rate=0.1`, `max_depth=60`, `subsample=1.0`, `gamma=10`, `reg_lambda=0`.

In [ ]:
log_loss(test_y, clf.predict_proba(test_X))

## Train (GPU)

In [ ]:
xgb.build_info()  # confirm CUDA/GPU support is compiled in

In [ ]:
!nvidia-smi

In [ ]:
# del train_X, valid_X, test_X

In [ ]:
del clf_gpu  # noqa forward reference

In [ ]:
import gc

gc.collect()

In [ ]:
config["cat_order"] = "node"
config["n_estimators"] = 150
cat_config = {}
cat_config["enable_categorical"] = config["cat_order"] in ["node", "tree", "ohe"]

if config["cat_order"] in ["node", "tree"]:
    cat_config["max_cat_threshold"] = 1_000_000
    cat_config["max_cat_to_onehot"] = 10
    cat_config["cat_order"] = config["cat_order"]
    cat_config["cat_reg_lambda"] = 400
else:
    cat_config["max_cat_threshold"] = 64
    cat_config["max_cat_to_onehot"] = 4_294_967_295

In [ ]:
12800 * (22 / 8)

In [ ]:
with wandb.init(project="criteo_boost_gpu_categorical", config=config) as run:
    clf_gpu = xgb.XGBClassifier(
        device="gpu",
        tree_method="hist",
        early_stopping_rounds=10,
        reg_lambda=35200,
        max_bin=256,
        gamma=10,
        min_child_weight=1,
        subsample=1.0,
        n_estimators=config["n_estimators"],
        learning_rate=0.1,
        max_depth=60,
        callbacks=[WandbCallback(log_feature_importance=True)],
        **cat_config,
    )
    clf_gpu.fit(
        train_X,
        train_y,
        eval_set=[(train_X, train_y), (valid_X, valid_y)],
        verbose=2,
    )

In [ ]:
zz = clf_gpu.get_booster().save_config()

In [ ]:
import json

zz1 = json.loads(zz)

In [ ]:
df = clf_gpu.get_booster().trees_to_dataframe()

In [ ]:
df.to_parquet("xgboost_cat_tree_smooth.parquet")

In [ ]:
clf_gpu.save_model("xgboost_cat_tree_smooth.ubj")

In [ ]:
train_y.shape[0]

In [ ]:
log_loss(test_y, clf_gpu.predict_proba(test_X))

## Hyperparameter sweep

`train()` below is the wandb sweep-agent entry point: it reads hyperparameters from
`wandb.config`, fits an `XGBClassifier`, and logs to the run started by `wandb.agent`.

In [ ]:
def train():
    """wandb sweep agent entry point: trains an XGBClassifier for one sweep run."""
    config = {
        "sample_every": 1,
        # max_cat_to_onehot":,
        "n_estimators": 150,
        "learning_rate": 0.1,
        "gamma": 10,
        "min_child_weight": 1,
        "max_depth": 60,
        "device": "gpu",
        "tree_method": "hist",
        "early_stopping_rounds": 10,
        "max_bin": 256,
        "subsample": 1.0,
    }

    config["cat_order"] = "tree"
    config["cat_cutoff"] = 100
    config["reg_lambda"] = 100
    config["cat_pseudo_count"] = 16  # equivalent to cat_reg_lambda

    config["enable_categorical"] = config["cat_order"] in ["node", "tree", "ohe"]

    if config["cat_order"] in ["node", "tree"]:
        config["cat_reg_lambda"] = 400
        config["max_cat_threshold"] = 1_000_000
        config["max_cat_to_onehot"] = 10
    elif config["cat_order"] == "target":
        pass
    elif config["cat_order"] == "ohe":
        config["max_cat_threshold"] = 64
        config["max_cat_to_onehot"] = 4_294_967_295

    not_xgb_keys = ["sample_every", "cat_cutoff", "cat_pseudo_count"]
    if config["cat_order"] not in ["node", "tree"]:
        not_xgb_keys.append("cat_order")
    with wandb.init(config=config) as _run:
        config = wandb.config  # override with sweeps settings
        config_xgb = {
            key: value for key, value in config.items() if key not in not_xgb_keys
        }
        data_dir = get_data_dir()

        (train_X, train_y), (valid_X, valid_y), (_test_X, _test_y) = get_data(
            config, data_dir
        )
        clf = xgb.XGBClassifier(
            **config_xgb, callbacks=[WandbCallback(log_feature_importance=True)]
        )
        clf.fit(
            train_X,
            train_y,
            eval_set=[(train_X, train_y), (valid_X, valid_y)],
            verbose=2,
        )

In [ ]:
sweep_config = {
    "method": "grid",  # try grid or random
    "metric": {
        "name": "validation_1-logloss",
        "goal": "minimize",
    },
    "parameters": {
        "reg_lambda": {"values": [0, 1, 10, 100]},
        "min_child_weight": {"values": [0.5, 1, 2]},
    },
}

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="criteo_boost_gpu_categorical")

In [ ]:
wandb.agent(sweep_id, train)

In [ ]:
wandb.finish()